# Spectral Clustering Assignment
**Task:**
1. Identify the optimal number of clusters in the provided graph.
2. Print the cluster number for each node (0-599).

In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from scipy.sparse.csgraph import laplacian
from scipy.sparse.linalg import eigsh
from sklearn.cluster import SpectralClustering

## 1. Load Data and Construct Graph
We load the edges from the CSV and create an adjacency matrix.

In [ ]:
# The CSV contains a header 'Node 1, Node 2'
file_path = '/content/spectral_graph_600nodes_edges.csv'
edges_df = pd.read_csv(file_path)

# Construct graph using networkx
G = nx.from_pandas_edgelist(edges_df, source='Node 1', target='Node 2')
adj_matrix = nx.adjacency_matrix(G).toarray()

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

## 2. Identify Number of Clusters (Eigengap Heuristic)
We analyze the eigenvalues of the Laplacian matrix. A large 'gap' between consecutive eigenvalues indicates the optimal number of clusters ($k$).

In [ ]:
# Compute unnormalized Laplacian
L = laplacian(adj_matrix, normed=False)

# Calculate the smallest 10 eigenvalues
eigenvalues, _ = eigsh(L, k=10, which='SA')
eigenvalues = np.sort(eigenvalues)

# Identify the gap (skipping the first zero eigenvalue)
gaps = np.diff(eigenvalues)
optimal_k = np.argmax(gaps[1:]) + 2 

print(f"Optimal number of clusters identified: {optimal_k}")

# Optional Visualization of the eigengap
plt.figure(figsize=(8, 4))
plt.plot(range(1, len(eigenvalues) + 1), eigenvalues, marker='o')
plt.title('Laplacian Eigenvalues (Eigengap Analysis)')
plt.grid(True)
plt.show()

## 3. Perform Spectral Clustering and Print Results
We assign each node to a cluster and output the assignments.

In [ ]:
# Apply Spectral Clustering
sc = SpectralClustering(n_clusters=optimal_k, affinity='precomputed', random_state=42)
cluster_labels = sc.fit_predict(adj_matrix)

# Print cluster number for each node
print("Node Cluster Assignments:")
nodes = sorted(G.nodes())
for node in nodes:
    print(f"Node {node}: Cluster {cluster_labels[node]}")

## 4. Visual Summary
A spring layout visualization showing the clusters.

In [ ]:
plt.figure(figsize=(10, 7))
pos = nx.spring_layout(G, seed=42)
nx.draw(G, pos, node_color=cluster_labels, node_size=30, cmap='viridis', alpha=0.8)
plt.title(f"Spectral Clustering Result with k={optimal_k}")
plt.show()